# 3057. Employees Project Allocation

## Problem
We need to find employees who are allocated to projects with a **workload that exceeds the average workload of all employees in their respective teams**.  

- Return columns: `employee_id`, `project_id`, `employee_name`, `project_workload`.  
- Order the result by `employee_id`, `project_id` in ascending order.

---

## Schema

### Table: Project
| Column Name  | Type | Description                                      |
|--------------|------|--------------------------------------------------|
| project_id   | INT  | Project identifier                               |
| employee_id  | INT  | Foreign key referencing Employees                |
| workload     | INT  | Workload assigned to the employee for the project |

---

### Table: Employees
| Column Name  | Type     | Description                        |
|--------------|----------|------------------------------------|
| employee_id  | INT      | Primary key, unique employee ID     |
| name         | VARCHAR  | Employee name                      |
| team         | VARCHAR  | Team to which the employee belongs  |

---

## Sample Data

### Project
| project_id | employee_id | workload |
|------------|-------------|----------|
| 1          | 1           | 45       |
| 1          | 2           | 90       |
| 2          | 3           | 12       |
| 2          | 4           | 68       |

### Employees
| employee_id | name   | team |
|-------------|--------|------|
| 1           | Khaled | A    |
| 2           | Ali    | B    |
| 3           | John   | B    |
| 4           | Doe    | A    |

---

## Expected Output
| employee_id | project_id | employee_name | project_workload |
|-------------|------------|---------------|------------------|
| 2           | 1          | Ali           | 90               |
| 4           | 2          | Doe           | 68               |

---

## PySpark Code: Create DataFrames and Temp Views

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Schema for Project
project_schema = StructType([
    StructField("project_id", IntegerType(), False),
    StructField("employee_id", IntegerType(), False),
    StructField("workload", IntegerType(), False)
])

# Schema for Employees
employees_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("team", StringType(), False)
])

# Data for Project
project_data = [
    (1, 1, 45),
    (1, 2, 90),
    (2, 3, 12),
    (2, 4, 68)
]

# Data for Employees
employees_data = [
    (1, "Khaled", "A"),
    (2, "Ali", "B"),
    (3, "John", "B"),
    (4, "Doe", "A")
]

# Create DataFrames
project_df = spark.createDataFrame(project_data, project_schema)
employees_df = spark.createDataFrame(employees_data, employees_schema)

# Register Temp Views
project_df.createOrReplaceTempView("Project")
employees_df.createOrReplaceTempView("Employees")

# Quick check
project_df.show()
employees_df.show()


In [0]:
%sql
with cte as (
  Select avg(workload)over(partition by project_id ) as avg_workload ,workload    , employee_id  , project_id   from project
)
Select e.employee_id  ,   p.project_id  ,e.name as  employee_name  , p.workload as project_workload   
 from Employees  e left join cte p
on e.employee_id  = p.employee_id 
where p.workload     > p.avg_workload



In [0]:
%sql
with cte as (
  Select avg(workload)over(partition by project_id ) as avg_workload 
  ,p.workload as project_workload    , e.employee_id as employee_id  , p.project_id  as project_id
  ,e.name as  employee_name  
    from Employees  e left join project p
    on e.employee_id  = p.employee_id 
)
Select employee_id  ,   project_id  ,  employee_name  , project_workload   
 from  cte p
where p.project_workload     > p.avg_workload

# Employees Project Allocation – Documentation

## Objective
Identify employees whose project workload is greater than the **average workload** of all employees working on the same project.

---

## Approach 1: Average Workload First, Join Later
- A Common Table Expression (CTE) is created from the **Project** table.  
- Within the CTE, the average workload per project is calculated using a window function (`AVG(workload) OVER (PARTITION BY project_id)`), alongside each employee’s workload and project details.  
- The main query then joins this CTE with the **Employees** table on `employee_id`.  
- Finally, it filters employees whose workload exceeds the calculated project average.  

**Key idea:** Workload averages are computed first, then employee details are added later through a join.

---

## Approach 2: Join Employees and Projects First, Then Calculate
- The CTE begins by joining the **Employees** and **Project** tables directly.  
- After the join, the average workload per project is calculated using the same window function.  
- The CTE also keeps employee details (`employee_id`, `name`, `project_id`, workload).  
- The outer query simply filters rows where the project workload is greater than the average workload.  

**Key idea:** Employee and project data are combined first, and then averages are computed within the joined dataset.

---

## Difference Between the Two Approaches
- **Approach 1** separates workload calculation from employee details, making the logic easier to follow step by step.  
- **Approach 2** is more compact, as it combines the join and workload calculation in one step.  
- Both approaches produce the same result, but the structure and readability differ.

---

## Mistake Made
- I did not pay close attention to **column names** when writing the query.  
- Using the wrong column in the window function or join condition caused failures and incorrect results.  
- Lesson learned: Always read the question carefully and align column usage with the expected output.

---

## Takeaway
- Both approaches are valid for solving the problem.  
- The choice depends on whether you prefer **clarity (Approach 1)** or **conciseness (Approach 2)**.  
- Careful attention to column names and query requirements is essential to avoid errors.
